# PS S6E6 047: XGB multiclass FE/objective/fold variant

- Experiment: `exp_20260614_047_xgb_multiclass_fe_objective_foldvariant`
- Base notebook: `exp_20260608_030_ovr_xgb_as_is`
- Purpose: create a materially different XGB-family OOF/pred material for the post-046 pool.
- Main changes from 030:
  - Objective: binary one-vs-rest XGB -> multiclass `multi:softprob` XGB
  - Fold/seed: `SEED=42`, `SEED_LIST=[60, 0, 2809]` -> `SEED=2027`, `SEED_LIST=[2027, 3407, 7117]`
  - Feature Engineering: add richer color/magnitude/redshift/flux style features and additional quantile-bin categorical features
  - Target Encoding: replace binary TargetEncoder with fold-safe multiclass smoothed target-encoding for selected combo columns
- No external stacking files, no bias search for final output.
- Output filenames include exp_id and experiment number 047.


In [1]:
# ============================================================
# 0. Imports / config
# ============================================================
import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import balanced_accuracy_score, recall_score, log_loss, confusion_matrix
from xgboost import XGBClassifier

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260614_047_xgb_multiclass_fe_objective_foldvariant"
EXP_NO = "047"
SEED = 2027
SEED_LIST = [2027, 3407, 7117]
FOLDS = 5

ID = "id"
TARGET = "class"
CLASS_NAMES = ["GALAXY", "QSO", "STAR"]
target_mapping = {"GALAXY": 0, "QSO": 1, "STAR": 2}
reverse_mapping = {0: "GALAXY", 1: "QSO", 2: "STAR"}

NUMS = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
BANDS = ["u", "g", "r", "i", "z"]
CATS = ["spectral_type", "galaxy_population"]

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTDIR = WORKDIR
OUTDIR.mkdir(parents=True, exist_ok=True)

OOF_NPY = OUTDIR / f"oof_{EXP_ID}_proba.npy"
PRED_NPY = OUTDIR / f"pred_{EXP_ID}_proba.npy"
OOF_CSV = OUTDIR / f"oof_{EXP_ID}.csv"
PRED_CSV = OUTDIR / f"pred_{EXP_ID}.csv"
SUB_CSV = OUTDIR / f"submission_{EXP_ID}.csv"
CV_RESULT_JSON = OUTDIR / f"cv_result_{EXP_ID}.json"
FOLD_SCORES_CSV = OUTDIR / f"fold_scores_{EXP_ID}.csv"
FEATURE_INFO_JSON = OUTDIR / f"feature_info_{EXP_ID}.json"
FEATURE_COLUMNS_CSV = OUTDIR / f"feature_columns_{EXP_ID}.csv"
MODEL_LOG_CSV = OUTDIR / f"xgb_multiclass_model_log_{EXP_ID}.csv"

def seed_everything(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)

print("EXP_ID:", EXP_ID)
print("WORKDIR:", WORKDIR)
print("SEED_LIST:", SEED_LIST)


EXP_ID: exp_20260614_047_xgb_multiclass_fe_objective_foldvariant
WORKDIR: /kaggle/working
SEED_LIST: [2027, 3407, 7117]


In [2]:
# ============================================================
# 1. Load competition data
# ============================================================
candidate_dirs = [
    Path("/kaggle/input/competitions/playground-series-s6e6"),
    Path("/kaggle/input/playground-series-s6e6"),
    Path("/kaggle/input/playground-series-season-6-episode-6"),
]

DATA_DIR = None
for d in candidate_dirs:
    if (d / "train.csv").exists() and (d / "test.csv").exists() and (d / "sample_submission.csv").exists():
        DATA_DIR = d
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find train.csv/test.csv/sample_submission.csv under known Kaggle input paths.")

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(TRAIN_PATH, index_col=ID)
test = pd.read_csv(TEST_PATH, index_col=ID)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

train[TARGET] = train[TARGET].map(target_mapping).astype("int32")

print("DATA_DIR:", DATA_DIR)
print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)
print(train[TARGET].value_counts().sort_index())


DATA_DIR: /kaggle/input/competitions/playground-series-s6e6
train: (577347, 11)
test: (247435, 10)
sample_submission: (247435, 2)
class
0    377480
1    117143
2     82724
Name: count, dtype: int64


In [3]:
# ============================================================
# 2. Feature engineering: 030-based XGB FE variant
# ============================================================
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
num_cols = train.select_dtypes(exclude=["object"]).columns.tolist()
cat_cols = [col for col in cat_cols if col not in [ID, TARGET]]
num_cols = [col for col in num_cols if col not in [ID, TARGET]]

print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols))

category_map = {}
color_pairs = [
    ("u", "g"),
    ("g", "r"),
    ("r", "i"),
    ("i", "z"),
    ("u", "r"),
    ("g", "i"),
    ("r", "z"),
]

def _safe_div(a, b, eps: float = 1e-6):
    return a / (b + eps)

def feature_engineering(df: pd.DataFrame, fit: bool = False) -> pd.DataFrame:
    df = df.copy()

    # Color / magnitude interactions
    for a, b in color_pairs:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype("float32")
        df[f"_abs_{a}-{b}"] = (df[f"_{a}-{b}"].abs()).astype("float32")

    band_mat = df[BANDS].astype("float32")
    df["_mag_mean"] = band_mat.mean(axis=1).astype("float32")
    df["_mag_std"] = band_mat.std(axis=1).astype("float32")
    df["_mag_min"] = band_mat.min(axis=1).astype("float32")
    df["_mag_max"] = band_mat.max(axis=1).astype("float32")
    df["_mag_range"] = (df["_mag_max"] - df["_mag_min"]).astype("float32")
    df["_mag_slope_uz"] = ((df["z"] - df["u"]) / 4.0).astype("float32")
    df["_mag_curvature_gri"] = (df["g"] - 2.0 * df["r"] + df["i"]).astype("float32")

    # Redshift transforms and band/redshift ratios
    red_abs = df["redshift"].abs().astype("float32")
    df["_redshift_abs"] = red_abs
    df["_log1p_redshift_abs"] = np.log1p(red_abs).astype("float32")
    df["_redshift_sq"] = (df["redshift"] ** 2).astype("float32")
    df["_redshift_cbrt"] = np.cbrt(df["redshift"]).astype("float32")
    for b in BANDS:
        df[f"_{b}_div_redshift"] = _safe_div(df[b], df["redshift"]).astype("float32")
        df[f"_redshift_x_{b}"] = (df["redshift"] * df[b]).astype("float32")

    # Flux-style features; intentionally lightweight.
    for b in BANDS:
        df[f"_flux_{b}"] = np.power(10.0, -0.4 * df[b]).astype("float32")
    flux_cols = [f"_flux_{b}" for b in BANDS]
    flux_mat = df[flux_cols].astype("float32")
    df["_flux_mean"] = flux_mat.mean(axis=1).astype("float32")
    df["_flux_std"] = flux_mat.std(axis=1).astype("float32")
    df["_flux_range"] = (flux_mat.max(axis=1) - flux_mat.min(axis=1)).astype("float32")
    df["_flux_u_div_g"] = _safe_div(df["_flux_u"], df["_flux_g"]).astype("float32")
    df["_flux_g_div_r"] = _safe_div(df["_flux_g"], df["_flux_r"]).astype("float32")
    df["_flux_i_div_z"] = _safe_div(df["_flux_i"], df["_flux_z"]).astype("float32")

    # Categorize string categoricals
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype("int32")
        df[col] = codes
        df[col] = df[col].astype("category")

    # Categorize floored numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        floored = np.floor(df[col])
        if fit:
            codes, uniques = floored.factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = floored.map(code_map).fillna(-1).astype("int32")
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype("category")

    # Discretize selected numerical columns.
    # This intentionally differs from 030, which used only delta [100, 500].
    bin_config = {
        "alpha": [64],
        "delta": [64, 256],
        "redshift": [32, 128],
        "u": [64],
        "g": [64],
        "r": [64],
        "i": [64],
        "z": [64],
    }
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode="ordinal",
                    strategy="quantile",
                    subsample=None,
                )
                binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype("int32")
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype("category")

    return df

comb = feature_engineering(pd.concat([train.drop(columns=TARGET), test], ignore_index=True), fit=True)
y_series = train[TARGET].copy().reset_index(drop=True)
test = comb.iloc[len(y_series):].reset_index(drop=True)
train_fe = comb.iloc[:len(y_series)].reset_index(drop=True)
train_fe[TARGET] = y_series.copy()
del comb
gc.collect()

new_cat_cols = [col for col in train_fe.columns if col.endswith("_")]
new_num_cols = [col for col in train_fe.columns if col.startswith("_")]

DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)
if DROP:
    train_fe.drop(DROP, axis=1, inplace=True)
    test.drop(DROP, axis=1, inplace=True)
    new_cat_cols = [col for col in new_cat_cols if col not in DROP]

CATEGORY = CATS + [c for c in test.columns if "digit" in c]
for c in CATEGORY:
    if c not in train_fe.columns:
        continue
    freq = train_fe[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    train_fe[c] = train_fe[c].map(lambda x: mapping.get(x, mapping_default)).astype("int32")
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default)).astype("int32")

NUMS_EFFECTIVE = [c for c in NUMS if c in train_fe.columns]
CATS_EFFECTIVE = [c for c in CATS if c in train_fe.columns]

print("train_fe:", train_fe.shape)
print("test_fe:", test.shape)
print("new_cat_cols:", len(new_cat_cols))
print("new_num_cols:", len(new_num_cols))


init len(cat_cols): 2
init len(num_cols): 8
DROP: []
train_fe: (577347, 75)
test_fe: (247435, 74)
new_cat_cols: 18
new_num_cols: 46


In [4]:
# ============================================================
# 3. Combo features for fold-safe multiclass Target Encoding
# ============================================================
TE_columns = []

important_combos = [
    ("alpha_cat_", "delta_cat_"),
    ("u_cat_", "z_cat_"),
    ("g_cat_", "r_cat_"),
    ("redshift_128_quantile_bin_", "z_64_quantile_bin_"),
]
important_combos = sorted(important_combos)

for cols in important_combos:
    if not all(c in train_fe.columns and c in test.columns for c in cols):
        print("skip combo due to missing columns:", cols)
        continue

    name = "-".join(cols)

    train_fe[name] = train_fe[cols[0]].astype(str)
    for col in cols[1:]:
        train_fe[name] = train_fe[name] + "_" + train_fe[col].astype(str)

    test[name] = test[cols[0]].astype(str)
    for col in cols[1:]:
        test[name] = test[name] + "_" + test[col].astype(str)

    combined = pd.concat([train_fe[name], test[name]], ignore_index=True)
    combined_codes, _ = combined.factorize()

    if pd.Series(combined_codes).nunique() > len(combined_codes) // 2:
        train_fe = train_fe.drop(name, axis=1)
        test = test.drop(name, axis=1)
        print("drop high-card combo:", name)
        continue

    train_fe[name] = combined_codes[:len(train_fe)].astype("int32")
    test[name] = combined_codes[len(train_fe):len(train_fe) + len(test)].astype("int32")
    TE_columns.append(name)

print("TE_columns:", TE_columns)


TE_columns: ['alpha_cat_-delta_cat_', 'g_cat_-r_cat_', 'redshift_128_quantile_bin_-z_64_quantile_bin_', 'u_cat_-z_cat_']


In [5]:
# ============================================================
# 4. Prepare matrices
# ============================================================
train_full = train_fe.copy()
test_df = test.copy().reset_index(drop=True)

train_full["id"] = train_full.index
train_full[TARGET] = train_full[TARGET].map(reverse_mapping)
test_df["id"] = sample_submission["id"].values

y = train_full[TARGET].map(target_mapping).values.astype("int32")
target_ovr = pd.DataFrame({
    "GALAXY": (y == 0).astype("int32"),
    "QSO": (y == 1).astype("int32"),
    "STAR": (y == 2).astype("int32"),
})

cat_cols_final = [c for c in (CATS + new_cat_cols) if c in train_full.columns and c != TARGET]
feature_cols = [c for c in train_full.columns if c not in ["id", TARGET] and train_full[c].dtype != "object"]
feature_cols = cat_cols_final + [c for c in feature_cols if c not in cat_cols_final]

features = train_full[feature_cols].copy()
test_features = test_df[feature_cols].copy()
test_ids = test_df["id"].values

# Encode category dtypes as int codes consistently.
for c in cat_cols_final:
    if str(features[c].dtype) == "category":
        combined_cats = pd.concat([features[c], test_features[c]]).astype(str).unique()
        features[c] = pd.Categorical(features[c].astype(str), categories=combined_cats).codes.astype("int32")
        test_features[c] = pd.Categorical(test_features[c].astype(str), categories=combined_cats).codes.astype("int32")
    else:
        features[c] = features[c].astype("int32")
        test_features[c] = test_features[c].astype("int32")

feature_info = {
    "n_features": int(len(feature_cols)),
    "n_cat_cols_final": int(len(cat_cols_final)),
    "n_te_columns": int(len(TE_columns)),
    "feature_cols": feature_cols,
    "cat_cols_final": cat_cols_final,
    "te_columns": TE_columns,
    "source": "030-based XGB FE variant for multiclass softprob objective",
}

pd.DataFrame({"feature": feature_cols}).to_csv(FEATURE_COLUMNS_CSV, index=False)
with open(FEATURE_INFO_JSON, "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2, ensure_ascii=False, default=str)

print("features:", features.shape)
print("test_features:", test_features.shape)
print("Total features:", len(feature_cols))
print("TE columns:", TE_columns)


features: (577347, 78)
test_features: (247435, 78)
Total features: 78
TE columns: ['alpha_cat_-delta_cat_', 'g_cat_-r_cat_', 'redshift_128_quantile_bin_-z_64_quantile_bin_', 'u_cat_-z_cat_']


In [6]:
# ============================================================
# 5. XGB multiclass training body
# ============================================================
def xgb_params_for(seed: int) -> dict:
    return {
        "objective": "multi:softprob",
        "num_class": len(CLASS_NAMES),
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "device": "cuda",
        "n_estimators": 9000,
        "early_stopping_rounds": 350,
        "learning_rate": 0.025,
        "max_depth": 5,
        "min_child_weight": 2.0,
        "subsample": 0.82,
        "colsample_bytree": 0.62,
        "max_bin": 512,
        "reg_alpha": 0.05,
        "reg_lambda": 2.0,
        "random_state": seed,
    }

def add_multiclass_target_encoding(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: np.ndarray,
    te_cols,
    smoothing: float = 50.0,
):
    """Fold-safe validation/test TE.

    For each TE combo column and each class, fit smoothed class-rate encodings on
    the training fold only, then transform train/valid/test. This is intentionally
    simple and robust for XGB material generation.
    """
    if len(te_cols) == 0:
        return X_train, X_val, X_test

    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    y_train = pd.Series(y_train).reset_index(drop=True)

    for col in te_cols:
        train_key = X_train[col].reset_index(drop=True)

        for cls_id, cls_name in reverse_mapping.items():
            binary_target = (y_train == cls_id).astype("float32")
            prior = float(binary_target.mean())

            tmp = pd.DataFrame({
                "key": train_key.values,
                "target": binary_target.values,
            })
            stats = tmp.groupby("key")["target"].agg(["mean", "count"])
            enc = (stats["mean"] * stats["count"] + prior * smoothing) / (stats["count"] + smoothing)
            mapping = enc.to_dict()

            new_col = f"TE_{col}_{cls_name}"
            X_train[new_col] = X_train[col].map(mapping).fillna(prior).astype("float32")
            X_val[new_col] = X_val[col].map(mapping).fillna(prior).astype("float32")
            X_test[new_col] = X_test[col].map(mapping).fillna(prior).astype("float32")

    X_train = X_train.drop(list(te_cols), axis=1)
    X_val = X_val.drop(list(te_cols), axis=1)
    X_test = X_test.drop(list(te_cols), axis=1)

    return X_train, X_val, X_test

def fit_multiclass_xgb_for_seed(seed: int):
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)

    oof_seed = np.zeros((len(features), len(CLASS_NAMES)), dtype=np.float64)
    pred_seed = np.zeros((len(test_features), len(CLASS_NAMES)), dtype=np.float64)
    logs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(features, y), start=1):
        fold_seed = seed + fold * 17

        X_train = features.iloc[tr_idx].reset_index(drop=True)
        X_val = features.iloc[val_idx].reset_index(drop=True)
        X_test = test_features.copy()

        y_train = y[tr_idx]
        y_val = y[val_idx]

        # Fold-safe multiclass TE for selected combo features.
        X_train, X_val, X_test = add_multiclass_target_encoding(
            X_train=X_train,
            X_val=X_val,
            X_test=X_test,
            y_train=y_train,
            te_cols=TE_columns,
            smoothing=50.0,
        )

        X_train.columns = X_train.columns.astype(str)
        X_val.columns = X_val.columns.astype(str)
        X_test.columns = X_test.columns.astype(str)

        print(f"\n[seed={seed}] fold={fold} fold_seed={fold_seed}")
        model = XGBClassifier(**xgb_params_for(fold_seed))
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=1000,
        )

        val_pred = model.predict_proba(X_val)
        test_pred = model.predict_proba(X_test)

        oof_seed[val_idx] = val_pred
        pred_seed += test_pred / FOLDS

        fold_bacc = balanced_accuracy_score(y_val, np.argmax(val_pred, axis=1))
        fold_logloss = log_loss(y_val, val_pred, labels=[0, 1, 2])
        best_iteration = getattr(model, "best_iteration", None)
        logs.append({
            "seed": seed,
            "fold_seed": fold_seed,
            "fold": fold,
            "balanced_accuracy": float(fold_bacc),
            "mlogloss": float(fold_logloss),
            "best_iteration": None if best_iteration is None else int(best_iteration),
            "n_train": int(len(tr_idx)),
            "n_valid": int(len(val_idx)),
            "n_features_after_te": int(X_train.shape[1]),
        })
        print(f"fold balanced_accuracy={fold_bacc:.9f}, mlogloss={fold_logloss:.6f}, best_iteration={best_iteration}")

        del model, X_train, X_val, X_test, y_train, y_val
        gc.collect()

    seed_cv = balanced_accuracy_score(y, np.argmax(oof_seed, axis=1))
    print(f"Seed {seed} OOF Balanced Accuracy: {seed_cv:.9f}")
    return oof_seed, pred_seed, logs

all_oof_by_seed = []
all_pred_by_seed = []
model_logs = []

for seed in SEED_LIST:
    print("=" * 80)
    print("SEED:", seed)
    oof_seed, pred_seed, logs = fit_multiclass_xgb_for_seed(seed)
    all_oof_by_seed.append(oof_seed)
    all_pred_by_seed.append(pred_seed)
    model_logs.extend(logs)

oof_proba = np.mean(all_oof_by_seed, axis=0)
pred_proba = np.mean(all_pred_by_seed, axis=0)

# Normalize defensively after seed averaging.
oof_proba = oof_proba / np.clip(oof_proba.sum(axis=1, keepdims=True), 1e-12, None)
pred_proba = pred_proba / np.clip(pred_proba.sum(axis=1, keepdims=True), 1e-12, None)

model_log_df = pd.DataFrame(model_logs)
model_log_df.to_csv(MODEL_LOG_CSV, index=False)

overall_cv = balanced_accuracy_score(y, np.argmax(oof_proba, axis=1))
print("\nOverall OOF Balanced Accuracy:", overall_cv)
display(model_log_df.head())
display(model_log_df.groupby(["seed"])["balanced_accuracy"].mean().reset_index())


SEED: 2027

[seed=2027] fold=1 fold_seed=2044
[0]	validation_0-mlogloss:0.84880
[1000]	validation_0-mlogloss:0.08719
[2000]	validation_0-mlogloss:0.08576
[2859]	validation_0-mlogloss:0.08572


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [03:59:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold balanced_accuracy=0.958749026, mlogloss=0.085644, best_iteration=2509

[seed=2027] fold=2 fold_seed=2061
[0]	validation_0-mlogloss:0.84894
[1000]	validation_0-mlogloss:0.08581
[2000]	validation_0-mlogloss:0.08417
[2933]	validation_0-mlogloss:0.08404
fold balanced_accuracy=0.960268579, mlogloss=0.083991, best_iteration=2583

[seed=2027] fold=3 fold_seed=2078
[0]	validation_0-mlogloss:0.84864
[1000]	validation_0-mlogloss:0.08453
[2000]	validation_0-mlogloss:0.08307
[2912]	validation_0-mlogloss:0.08296
fold balanced_accuracy=0.959646595, mlogloss=0.082921, best_iteration=2562

[seed=2027] fold=4 fold_seed=2095
[0]	validation_0-mlogloss:0.84868
[1000]	validation_0-mlogloss:0.08546
[2000]	validation_0-mlogloss:0.08411
[2772]	validation_0-mlogloss:0.08409
fold balanced_accuracy=0.959936587, mlogloss=0.084021, best_iteration=2422

[seed=2027] fold=5 fold_seed=2112
[0]	validation_0-mlogloss:0.84960
[1000]	validation_0-mlogloss:0.08774
[2000]	validation_0-mlogloss:0.08657
[2264]	validation

,seed,fold_seed,fold,balanced_accuracy,mlogloss,best_iteration,n_train,n_valid,n_features_after_te
0,2027,2044,1,0.958749,0.085644,2509,461877,115470,86
1,2027,2061,2,0.960269,0.083991,2583,461877,115470,86
2,2027,2078,3,0.959647,0.082921,2562,461878,115469,86
3,2027,2095,4,0.959937,0.084021,2422,461878,115469,86
4,2027,2112,5,0.958334,0.086550,1914,461878,115469,86


,seed,balanced_accuracy
0,2027,0.959387
1,3407,0.959378
2,7117,0.959217


In [7]:
# ============================================================
# 6. CV diagnostics / save outputs
# ============================================================
oof_pred_label = np.argmax(oof_proba, axis=1)
recalls = recall_score(y, oof_pred_label, average=None)
cm = confusion_matrix(y, oof_pred_label)

eval_skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_records = []
for fold, (_, val_idx) in enumerate(eval_skf.split(oof_proba, y), start=1):
    fold_score = balanced_accuracy_score(y[val_idx], oof_pred_label[val_idx])
    fold_recalls = recall_score(y[val_idx], oof_pred_label[val_idx], average=None)
    fold_records.append({
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        "recall_GALAXY": float(fold_recalls[0]),
        "recall_QSO": float(fold_recalls[1]),
        "recall_STAR": float(fold_recalls[2]),
        "n_valid": int(len(val_idx)),
    })

fold_scores = pd.DataFrame(fold_records)
fold_std = float(fold_scores["balanced_accuracy"].std(ddof=0))
fold_scores.to_csv(FOLD_SCORES_CSV, index=False)

np.save(OOF_NPY, oof_proba)
np.save(PRED_NPY, pred_proba)

oof_df = pd.DataFrame(oof_proba, columns=CLASS_NAMES)
oof_df.insert(0, ID, train_full["id"].values)
oof_df[TARGET] = [reverse_mapping[v] for v in y]
oof_df.to_csv(OOF_CSV, index=False)

pred_df = pd.DataFrame(pred_proba, columns=CLASS_NAMES)
pred_df.insert(0, ID, test_ids)
pred_df.to_csv(PRED_CSV, index=False)

test_pred_label = np.argmax(pred_proba, axis=1)
test_labels = [reverse_mapping[p] for p in test_pred_label]
submission = pd.DataFrame({ID: test_ids, TARGET: test_labels})
submission.to_csv(SUB_CSV, index=False)

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "created_at": "2026-06-14",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "model_family": "XGBoost",
    "model_type": "xgb_multiclass_softprob_fe_objective_foldvariant",
    "model": "XGBClassifier multiclass softprob",
    "source": "exp_20260608_030_ovr_xgb_as_is",
    "purpose": "Create a third-candidate XGB-family material with different FE, objective, seeds, and folds from 030.",
    "seed": SEED,
    "seed_list": SEED_LIST,
    "n_splits": FOLDS,
    "overall_cv": float(overall_cv),
    "fold_std": fold_std,
    "recalls": {
        "GALAXY": float(recalls[0]),
        "QSO": float(recalls[1]),
        "STAR": float(recalls[2]),
    },
    "confusion_matrix": cm.tolist(),
    "variant_design": {
        "base_exp_id": "exp_20260608_030_ovr_xgb_as_is",
        "changed_from_030": [
            "objective: binary one-vs-rest -> multi:softprob",
            "eval_metric: auc -> mlogloss",
            "seed/folds: SEED=2027 and seed_list=[2027,3407,7117]",
            "feature engineering: richer color/magnitude/redshift/flux features",
            "quantile bins: add alpha/redshift/band bins and alter delta bins",
            "target encoding: fold-safe multiclass smoothed TE for selected combos",
        ],
    },
    "xgb_params_base": xgb_params_for(SEED_LIST[0]),
    "fold_scores": fold_scores.to_dict(orient="records"),
    "model_log": model_log_df.to_dict(orient="records"),
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "submission": SUB_CSV.name,
        "fold_scores": FOLD_SCORES_CSV.name,
        "feature_info": FEATURE_INFO_JSON.name,
        "feature_columns": FEATURE_COLUMNS_CSV.name,
        "model_log": MODEL_LOG_CSV.name,
    },
}

with open(CV_RESULT_JSON, "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False, default=str)

print("overall_cv:", overall_cv)
print("fold_std:", fold_std)
print("recalls:", dict(zip(CLASS_NAMES, recalls)))
print("saved:", OOF_NPY.name, PRED_NPY.name, SUB_CSV.name)
display(fold_scores)
display(submission.head())


overall_cv: 0.9596874103404002
fold_std: 0.0007985906056383474
recalls: {'GALAXY': np.float64(0.9797976051711349), 'QSO': np.float64(0.9645902870850157), 'STAR': np.float64(0.93467433876505)}
saved: oof_exp_20260614_047_xgb_multiclass_fe_objective_foldvariant_proba.npy pred_exp_20260614_047_xgb_multiclass_fe_objective_foldvariant_proba.npy submission_exp_20260614_047_xgb_multiclass_fe_objective_foldvariant.csv


,fold,balanced_accuracy,recall_GALAXY,recall_QSO,recall_STAR,n_valid
0,1,0.958649,0.980105,0.962568,0.933273,115470
1,2,0.960613,0.979363,0.965940,0.936537,115470
2,3,0.960158,0.979933,0.966366,0.934176,115469
3,4,0.960204,0.979694,0.964743,0.936174,115469
4,5,0.958813,0.979893,0.963334,0.933212,115469


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [8]:
# ============================================================
# 7. Update registry / memo
# ============================================================
registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "XGBoost",
    "feature_family": "xgb_multiclass_fe_objective_foldvariant",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(overall_cv),
    "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": False,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(SUB_CSV),
    "role": "xgb_family_lower_correlation_material",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_corr",
    "notes": (
        "030-based XGB third-candidate variant. "
        "Changes OVR binary objective to multiclass softprob, changes seed/fold, "
        "adds richer FE and multiclass fold-safe TE. "
        "Main purpose is to obtain a materially different XGB-family OOF/pred source for 048 stacker/blend checks."
    ),
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

submission_counts = submission[TARGET].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
submission_ratios = (submission_counts / len(submission)).round(8)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "XGB multiclass FE/objective/fold variant",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "XGBClassifier multiclass softprob",
        "created_at": "2026-06-14",
    },
    "objective": {
        "summary": (
            "Run an XGB-family third-candidate experiment based on 030, but make it materially different "
            "by changing objective, FE, seed/fold split, and TE handling."
        ),
        "success_criteria": [
            "use 030 as implementation base",
            "change binary one-vs-rest objective to multiclass softprob",
            "use different seed/fold configuration",
            "add richer FE and additional qbin features",
            "save OOF proba npy with exp_id in filename",
            "save test pred proba npy with exp_id in filename",
            "save submission",
            "save cv_result",
            "save fold_scores",
            "update oof_registry",
            "write memo.yml",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "target_col": TARGET,
        "id_col": ID,
        "use_original": False,
    },
    "seed": {
        "base_seed": SEED,
        "seed_list": SEED_LIST,
        "numpy_seed": SEED,
        "xgb_random_state": "fold_seed = seed + fold * 17",
        "stratified_kfold_random_state": "each seed in seed_list",
    },
    "features": {
        "feature_family": "xgb_multiclass_fe_objective_foldvariant",
        "source_notebook": "exp_20260608_030_ovr_xgb_as_is",
        "feature_info": feature_info,
        "target_encoder_safety": (
            "For selected combo features, smoothed class-rate encodings are fit on each training fold only "
            "and then applied to validation/test. 042/044-style direct leakage is avoided for validation."
        ),
    },
    "model": {
        "family": "XGBoost",
        "type": "multiclass softprob",
        "class_order": CLASS_NAMES,
        "xgb_params_base": xgb_params_for(SEED_LIST[0]),
        "n_models": int(len(SEED_LIST) * FOLDS),
        "probability_postprocess": (
            "Average multiclass softprob predictions across seeds/folds, then normalize defensively so each row sums to 1."
        ),
        "changed_from_030": [
            "OVR binary XGB -> multiclass softprob XGB",
            "binary AUC monitoring -> multiclass logloss monitoring",
            "SEED 42 -> 2027",
            "SEED_LIST [60,0,2809] -> [2027,3407,7117]",
            "richer FE and additional qbins",
            "fold-safe multiclass smoothed TE instead of binary TargetEncoder",
        ],
    },
    "cv": {
        "scheme": "StratifiedKFold multiclass per seed",
        "n_splits": FOLDS,
        "shuffle": True,
        "score": float(overall_cv),
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
        "recalls": {
            "GALAXY": float(recalls[0]),
            "QSO": float(recalls[1]),
            "STAR": float(recalls[2]),
        },
        "model_log_path": MODEL_LOG_CSV.name,
    },
    "submission_distribution": {
        "total_rows": int(len(submission)),
        "class_counts": {k: int(v) for k, v in submission_counts.to_dict().items()},
        "class_ratio": {k: float(v) for k, v in submission_ratios.to_dict().items()},
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUB_CSV.name,
        "cv_result": CV_RESULT_JSON.name,
        "fold_scores": FOLD_SCORES_CSV.name,
        "feature_info": FEATURE_INFO_JSON.name,
        "feature_columns": FEATURE_COLUMNS_CSV.name,
        "model_log": MODEL_LOG_CSV.name,
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "pending_public_lb_and_corr",
        "reason": (
            "This is not expected to be a standalone final model. Its purpose is to produce a lower-correlation "
            "XGB-family OOF/pred material after RealMLP seed/fold variants failed to improve Public LB."
        ),
        "next_action": [
            f"Submit {SUB_CSV.name}",
            "Record Public LB",
            "Add OOF/pred npy to npy-files dataset",
            "Compare OOF correlation against 030, 040, 045, 046, and 033",
            "If correlation/diversity is favorable, run 048 GPU LogReg stacker add047 or blend check",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())


registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260614_047_xgb_multiclass_fe_objective_f...,XGBoost,xgb_multiclass_fe_objective_foldvariant,balanced_accuracy_score,0.959687,0.000799,NaN,False,True,False,/kaggle/working/oof_exp_20260614_047_xgb_multi...,/kaggle/working/pred_exp_20260614_047_xgb_mult...,/kaggle/working/submission_exp_20260614_047_xg...,xgb_family_lower_correlation_material,completed,pending_public_lb_and_corr,030-based XGB third-candidate variant. Changes...
